[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/customer-service-bot-llm-course/blob/main/Assignment4_Customer_Service_Chatbot.ipynb)

# Assignment 4: Context-Aware Customer Service Chatbot

**PSYC 51.17: Models of Language and Communication**

**Due Date: February 16, 2026 at 11:59 PM EST**

---

## Overview

In this assignment, you will build a customer service chatbot that uses **retrieval-augmented generation (RAG)** to answer questions about a fictional company called TechCo. Your system will:

1. **Retrieve** relevant FAQ entries using semantic search (sentence embeddings + cosine similarity)
2. **Compare** semantic search against a TF-IDF keyword baseline
3. **Generate** helpful responses based on the retrieved context

This assignment directly applies concepts from Lecture 17 (RAG) and builds on your knowledge of embeddings from Lectures 11-12.

### Learning Objectives

- Apply sentence embeddings for semantic search
- Implement cosine similarity retrieval (the same approach from Lecture 17)
- Compare semantic search vs. keyword matching baselines
- Build a working RAG-style chatbot
- Evaluate retrieval quality using Accuracy@k and MRR

---

## Table of Contents

1. [Setup and Installation](#1-setup-and-installation)
2. [Load Knowledge Base](#2-load-knowledge-base)
3. [Semantic Search](#3-semantic-search)
4. [TF-IDF Baseline](#4-tf-idf-baseline)
5. [Response Generation](#5-response-generation)
6. [Evaluation](#6-evaluation)
7. [Error Analysis and Examples](#7-error-analysis-and-examples)
8. [Interactive Demo](#8-interactive-demo)
9. [Bonus: Multi-Turn Conversation](#9-bonus-multi-turn-conversation) (Optional)
10. [Bonus: LLM-Based Generation](#10-bonus-llm-based-generation) (Optional)
11. [Reflection](#11-reflection)

---

## 1. Setup and Installation

In [ ]:
# Install required packages (run this cell first)
!pip install -q sentence-transformers torch
!pip install -q scikit-learn pandas numpy
!pip install -q matplotlib seaborn

In [ ]:
# Core imports
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
import random
random.seed(42)
np.random.seed(42)

# Check for GPU (optional, CPU works fine for this assignment)
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU (this is fine for this assignment)")

---

## 2. Load Knowledge Base

We provide a knowledge base of 50 FAQ entries for a fictional company called TechCo. The knowledge base covers account access, billing, product features, shipping and returns, technical support, and general information.

We also provide 20 test queries with ground-truth relevant FAQ IDs for evaluation.

In [ ]:
# Download the knowledge base and test queries
import urllib.request

KB_URL = "https://raw.githubusercontent.com/ContextLab/customer-service-bot-llm-course/main/knowledge_base.json"
TEST_URL = "https://raw.githubusercontent.com/ContextLab/customer-service-bot-llm-course/main/test_queries.json"

print("Downloading knowledge base...")
urllib.request.urlretrieve(KB_URL, "knowledge_base.json")

print("Downloading test queries...")
urllib.request.urlretrieve(TEST_URL, "test_queries.json")

# Load the data
with open("knowledge_base.json", "r") as f:
    knowledge_base = json.load(f)

with open("test_queries.json", "r") as f:
    test_queries = json.load(f)

print(f"Loaded {len(knowledge_base)} FAQ entries")
print(f"Loaded {len(test_queries)} test queries")

In [ ]:
# Explore the knowledge base
print("=== Sample FAQ Entry ===")
print(json.dumps(knowledge_base[0], indent=2))

print("\n=== Categories in Knowledge Base ===")
categories = {}
for entry in knowledge_base:
    cat = entry["category"]
    categories[cat] = categories.get(cat, 0) + 1
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count} entries")

In [ ]:
# Explore a test query
print("=== Sample Test Query ===")
print(json.dumps(test_queries[0], indent=2))

---

## 3. Semantic Search

Implement semantic search using sentence embeddings. This follows the approach from Lecture 17:

1. Load a sentence transformer model
2. Embed all FAQ questions
3. For a user query, embed it and find the most similar FAQ entries using cosine similarity

### 3.1 Load the Embedding Model and Encode Knowledge Base

In [ ]:
# Load sentence transformer model (provided)
from sentence_transformers import SentenceTransformer, util

print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded with embedding dimension: {embedder.get_sentence_embedding_dimension()}")

In [ ]:
# Encode all FAQ questions (provided)
kb_questions = [entry["question"] for entry in knowledge_base]
kb_ids = [entry["id"] for entry in knowledge_base]

print(f"Encoding {len(kb_questions)} FAQ questions...")
kb_embeddings = embedder.encode(kb_questions, convert_to_tensor=True, show_progress_bar=True)
print(f"Embeddings shape: {kb_embeddings.shape}")

### 3.2 Implement Semantic Search Function

**Your task:** Implement the `semantic_search` function below.

Use `util.cos_sim()` from sentence-transformers to compute cosine similarity.

In [ ]:
def semantic_search(query: str, top_k: int = 5) -> List[Dict]:
    """
    Find the top-k most similar FAQ entries for a query using semantic search.
    
    Args:
        query: The user's question (string)
        top_k: Number of results to return
    
    Returns:
        List of dicts, each containing:
            - 'id': FAQ entry ID
            - 'question': The FAQ question
            - 'answer': The FAQ answer  
            - 'score': Cosine similarity score (float)
    """
    # TODO: Implement semantic search
    # Step 1: Encode the query
    # Step 2: Compute cosine similarity with util.cos_sim()
    # Step 3: Get top-k indices
    # Step 4: Build results list
    raise NotImplementedError("Implement semantic_search")

In [ ]:
# Test your implementation (uncomment after implementing)
# test_query = "I forgot my password"
# results = semantic_search(test_query, top_k=3)
# print(f"Query: {test_query}\n")
# for i, r in enumerate(results, 1):
#     print(f"{i}. [{r['score']:.3f}] {r['question']}")

---

## 4. TF-IDF Baseline

Implement a TF-IDF keyword matching baseline to compare against semantic search.

### 4.1 Set Up TF-IDF

In [ ]:
# Set up TF-IDF vectorizer (provided)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vectorizer = TfidfVectorizer(stop_words='english', lowercase=True)
tfidf_matrix = tfidf_vectorizer.fit_transform(kb_questions)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

### 4.2 Implement TF-IDF Search Function

In [ ]:
def tfidf_search(query: str, top_k: int = 5) -> List[Dict]:
    """
    Find the top-k most similar FAQ entries using TF-IDF keyword matching.
    
    Args:
        query: The user's question (string)
        top_k: Number of results to return
    
    Returns:
        List of dicts with same format as semantic_search
    """
    # TODO: Implement TF-IDF search
    # Step 1: Transform query with tfidf_vectorizer.transform([query])
    # Step 2: Compute cosine_similarity against tfidf_matrix
    # Step 3: Get top-k indices
    # Step 4: Build results list
    raise NotImplementedError("Implement tfidf_search")

In [ ]:
# Test your implementation (uncomment after implementing)
# results = tfidf_search("I forgot my password", top_k=3)
# for i, r in enumerate(results, 1):
#     print(f"{i}. [{r['score']:.3f}] {r['question']}")

---

## 5. Response Generation

Create a function that generates helpful responses based on retrieved FAQ entries.

In [ ]:
def generate_response(query: str, search_results: List[Dict], 
                      high_confidence: float = 0.7,
                      low_confidence: float = 0.4) -> str:
    """
    Generate a helpful response based on retrieved FAQ entries.
    
    Behavior:
        - If top score >= high_confidence: Return the answer directly
        - If top score >= low_confidence: Offer top 2-3 as options
        - If top score < low_confidence: Return a fallback message
    """
    # TODO: Implement response generation
    raise NotImplementedError("Implement generate_response")

In [ ]:
# Test your implementation (uncomment after implementing)
# for q in ["How do I reset my password?", "Do you sell purple elephants?"]:
#     results = semantic_search(q, top_k=3)
#     response = generate_response(q, results)
#     print(f"User: {q}")
#     print(f"Bot: {response}\n")

---

## 6. Evaluation

Evaluate retrieval quality using Accuracy@k and MRR.

In [ ]:
def evaluate_retrieval(search_fn, test_data: List[Dict], k_values: List[int] = [1, 3, 5]) -> Dict:
    """
    Evaluate a search function on the test queries.
    
    Returns dict with 'accuracy@1', 'accuracy@3', 'accuracy@5', and 'mrr'
    """
    # TODO: Implement evaluation
    raise NotImplementedError("Implement evaluate_retrieval")

In [ ]:
# Run evaluation (uncomment after implementing)
# print("Evaluating Semantic Search...")
# semantic_results = evaluate_retrieval(semantic_search, test_queries)
# print(f"  Accuracy@1: {semantic_results['accuracy@1']:.2%}")
# print(f"  MRR: {semantic_results['mrr']:.3f}")
#
# print("\nEvaluating TF-IDF Baseline...")
# tfidf_results = evaluate_retrieval(tfidf_search, test_queries)
# print(f"  Accuracy@1: {tfidf_results['accuracy@1']:.2%}")
# print(f"  MRR: {tfidf_results['mrr']:.3f}")

In [ ]:
# Create comparison bar chart (uncomment and fill in after evaluation)
# metrics = ['Accuracy@1', 'Accuracy@3', 'Accuracy@5', 'MRR']
# semantic_scores = [...]  # Fill in from semantic_results
# tfidf_scores = [...]  # Fill in from tfidf_results
#
# x = np.arange(len(metrics))
# width = 0.35
# fig, ax = plt.subplots(figsize=(10, 6))
# ax.bar(x - width/2, semantic_scores, width, label='Semantic Search', color='#00693e')
# ax.bar(x + width/2, tfidf_scores, width, label='TF-IDF Baseline', color='#267aba')
# ax.set_ylabel('Score')
# ax.set_title('Semantic Search vs. TF-IDF Baseline')
# ax.set_xticks(x)
# ax.set_xticklabels(metrics)
# ax.legend()
# plt.show()

---

## 7. Error Analysis and Examples

### 7.1 Side-by-Side Comparison Table

In [ ]:
# TODO: Create side-by-side comparison showing rank of correct answer for each method

### 7.2 Examples Where Semantic Search Wins (5 examples)

In [ ]:
# TODO: Show 5 examples where semantic search outperforms TF-IDF

### 7.3 Examples Where TF-IDF Wins or Both Fail (3 examples)

In [ ]:
# TODO: Show 3 examples where TF-IDF wins or both fail

### 7.4 Analysis Paragraph (150-250 words)

*TODO: Write your analysis here*

---

## 8. Interactive Demo

In [ ]:
def chat():
    """Interactive chat interface for the customer service bot."""
    # TODO: Implement chat loop
    raise NotImplementedError("Implement chat")

# Uncomment to run: chat()

---

## 9. Bonus: Multi-Turn Conversation (Optional, +5 points)

In [ ]:
class ConversationManager:
    """BONUS: Manages multi-turn conversation context."""
    def __init__(self):
        self.history = []
    
    def add_turn(self, user_query, bot_response, retrieved_ids):
        pass  # TODO: Implement
    
    def get_context_enhanced_query(self, current_query):
        pass  # TODO: Implement

---

## 10. Bonus: LLM-Based Generation (Optional, +5 points)

In [ ]:
# BONUS: Use FLAN-T5 for generation
# !pip install -q transformers
# from transformers import pipeline
# generator = pipeline("text2text-generation", model="google/flan-t5-small")
#
# def generate_llm_response(query, search_results):
#     """BONUS: Generate response using FLAN-T5."""
#     pass  # TODO: Implement

---

## 11. Reflection (300-500 words)

Address these prompts:
1. How does semantic search change the customer service experience compared to keyword matching?
2. What are the limitations of your system?
3. How does this compare to the rule-based ELIZA chatbot from Assignment 1?
4. What role does the generation step play in RAG?

*TODO: Write your reflection here*

---

## Submission Checklist

- [ ] All required functions implemented
- [ ] Evaluation complete with bar chart
- [ ] Error analysis with 8+ examples
- [ ] Analysis paragraph written
- [ ] Interactive demo works
- [ ] Reflection written (300-500 words)
- [ ] All cells execute without errors